## Description
```
Problem        : Regression
Algorithm      : Logistic
Formula        : Weightes-Sum
Action-FN      : Sigmoid
Loss-FN        : Cross-Entropy-Binary
Hyperparameter : - - -
Train          : Supervised
Input          : Feature and label
Output         : Probability
Dataset        : Structured : housesInfo
```

## Import

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, zero_one_loss
from sklearn.metrics import mean_absolute_percentage_error

## Import dataset

In [2]:
dataset = pd.read_csv('/Volumes/data/documents/ai_document/file/dataset/housesInfo.txt', sep=" ", header=None, names=["bedrooms", "bathrooms", "area", "zipcode", "price"])

In [3]:
rows, cols = dataset.shape
print(dataset.head(5), '\n')
print("rows: {}, cols:{}".format(rows, cols))

   bedrooms  bathrooms  area  zipcode   price
0         4        4.0  4053    85255  869500
1         4        3.0  3343    36372  865200
2         3        4.0  3923    85266  889000
3         5        5.0  4022    85262  910000
4         3        4.0  4116    85266  971226 

rows: 535, cols:5


## Extract zipcode and count

In [4]:
zipcodes = dataset["zipcode"].value_counts().keys().tolist()
counts = dataset["zipcode"].value_counts().tolist()

In [7]:
print(zipcodes)
print(counts)

[92276, 93510, 93446, 92880, 94501, 91901, 92677, 94531, 96019, 85255, 92021, 85266, 93111, 81524, 95220, 92802, 85262, 62234, 62214, 98021, 85377, 91752, 60002, 81418, 62025, 92253, 60016, 92692, 90265, 62034, 62088, 91915, 94565, 95008, 90803, 90038, 93314, 93720, 93924, 92040, 90211, 94568, 92543, 62249, 85331, 93105, 60046, 36372, 81521]
[100, 60, 54, 49, 41, 32, 26, 22, 12, 12, 11, 11, 11, 11, 10, 9, 9, 7, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Remove count < 25

In [10]:
for (zipcode, count) in zip(zipcodes, counts):
  if count < 25:
    idxs = dataset[dataset["zipcode"] == zipcode].index
    dataset.drop(idxs, inplace=True)

In [11]:
zipcodes = dataset["zipcode"].value_counts().keys().tolist()
counts = dataset["zipcode"].value_counts().tolist()
print(zipcodes)
print(counts)

[92276, 93510, 93446, 92880, 94501, 91901, 92677]
[100, 60, 54, 49, 41, 32, 26]


In [12]:
rows, cols = dataset.shape
print(dataset.head(5), '\n')
print("rows: {}, cols:{}".format(rows, cols))

    bedrooms  bathrooms  area  zipcode   price
30         5        3.0  2520    93446  789000
32         3        2.0  1802    93446  365000
39         3        3.0  2146    93446  455000
80         4        2.5  2464    91901  599000
81         2        2.0  1845    91901  529800 

rows: 362, cols:5


## Separate data and label

In [13]:
x = dataset.iloc[:, :4]
y = dataset.iloc[:, 4]

In [14]:
print(x.head(5))
print(y.head(5))

    bedrooms  bathrooms  area  zipcode
30         5        3.0  2520    93446
32         3        2.0  1802    93446
39         3        3.0  2146    93446
80         4        2.5  2464    91901
81         2        2.0  1845    91901
30    789000
32    365000
39    455000
80    599000
81    529800
Name: price, dtype: int64


## Separate tarin and test

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [16]:
print("x_train: {}".format(x_train.shape))
print("x_test: {}".format(x_test.shape))
print("y_train: {}".format(y_train.shape))
print("y_test: {}".format(y_test.shape))

x_train: (289, 4)
x_test: (73, 4)
y_train: (289,)
y_test: (73,)


## Normalizing continuous features

In [17]:
continuous = ["bedrooms", "bathrooms", "area"]
sc = StandardScaler()
x_train_continuous = sc.fit_transform(x_train[continuous])
x_test_continuous = sc.fit_transform(x_test[continuous])

## Encoding categorical features : W1

In [18]:
encdoer = OneHotEncoder(sparse_output=False)
x_train_categorical = encdoer.fit_transform(np.array(x_train["zipcode"]).reshape(-1, 1))
x_test_categorical = encdoer.fit_transform(np.array(x_test["zipcode"]).reshape(-1, 1))

In [19]:
print(x_train_categorical)
print(x_test_categorical)

[[0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 ...
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]]
[[0. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0.

## Encoding categorical features : W2

In [20]:
encdoer = LabelBinarizer()
x_train_categorical = encdoer.fit_transform(x_train["zipcode"])
x_test_categorical = encdoer.fit_transform(x_test["zipcode"])

In [21]:
print(x_train_categorical)
print(x_test_categorical)

[[0 1 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 0 1]]
[[0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0]
 [1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1]
 [0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 0 0 1 0]
 [0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [1 0 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1]
 [0 0 1 0 0 0 0]
 [1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 1 0 0

## Concatenating continuous with categorical with features



In [22]:
x_train = np.hstack([x_train_continuous, x_train_categorical])
x_test = np.hstack([x_test_continuous, x_test_categorical])

## Normalizing label

In [23]:
maxPrice = y_train.max()
y_train_n = y_train / maxPrice
y_test_n = y_test / maxPrice

In [24]:
print("maxPrice:", maxPrice, '\n')
print(pd.DataFrame({"y_train": y_train, "y_train_n": y_train_n}))

maxPrice: 5858000 

     y_train  y_train_n
351    98900   0.016883
479   739900   0.126306
498   319000   0.054455
185  1495000   0.255207
194   625000   0.106692
..       ...        ...
181   490000   0.083646
228   579000   0.098839
412   139000   0.023728
490   439000   0.074940
212   599000   0.102253

[289 rows x 2 columns]


In [25]:
y_train = y_train_n
y_test = y_test_n

## Train

### LinearRegression

In [26]:
mdl = LinearRegression()
mdl.fit(x_train, y_train)

LinearRegression()

### SGDRegressor

In [27]:
mdl = SGDRegressor(tol=0.00000001)
mdl.fit(x_train, y_train)

SGDRegressor(tol=1e-08)

## Predict

In [28]:
y_pred = mdl.predict(x_test)
df_pred = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
df_pred["Correct"] = df_pred["Actual"] == df_pred["Predicted"]
print(df_pred)

       Actual  Predicted  Correct
305  0.087060   0.066389    False
110  0.114203   0.156690    False
92   0.167122   0.135615    False
464  0.131273   0.125316    False
145  0.110942   0.142348    False
..        ...        ...      ...
439  0.072550   0.057994    False
192  0.162001   0.126466    False
204  0.110789   0.101180    False
483  0.047798   0.062017    False
445  0.092352   0.121332    False

[73 rows x 3 columns]


## Evaluation

In [29]:
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

MSE: 0.0007880452906136211
MAE: 0.020558267454824593
RMSE: 0.028072144389298462


## Evaluation

In [ ]:
preds = mdl.predict(x_test)
diff = preds- y_test
percentDiff = (diff / y_test) * 100
absPercentDiff = np.abs(percentDiff)
mean = np.mean(absPercentDiff)
std = np.std(absPercentDiff)
print("[INFO] mean: {:.2f}%, std: {:.2f}%".format(mean, std))

[INFO] mean: 33.11%, std: 41.31%


## Evaluation

In [31]:
mape = mean_absolute_percentage_error(y_test, y_pred)
print("mape: {:.2f}%".format(mape))

mape: 0.33%
